# Phase D: Session & Capture Facade — Acceptance Notebook

This notebook exercises the Phase D public API.

**Sections 1–8** are offline (mocked or structural) and can run without hardware.
**Sections 9–10** require live hardware and should only be executed in the lab.

Final marker: `PHASE_D_SESSION_AND_CAPTURE_FACADE_PASS`

In [ ]:
from pathlib import Path

REPO_ROOT = Path(r"C:\Users\khams008\Documents\awr2944-fmcw-radar")
PROJECT_ROOT = Path(r"C:\Users\khams008\Documents\awr2944-live-project")

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

assert PROJECT_ROOT.is_dir()
assert (PROJECT_ROOT / "awr2944.toml").is_file()
assert (PROJECT_ROOT / ".awr2944" / "local.toml").is_file()
assert (PROJECT_ROOT / "profiles" / "smoke_v1.toml").is_file()
print("Live project paths validated successfully.")

In [ ]:
from pathlib import Path
import sys, tempfile, warnings
sys.path.insert(0, str(REPO_ROOT / 'src'))

from awr2944_dca import RadarProject, RadarProfile

## 1. Create Test Project & Verify Profile Integration

In [ ]:
with tempfile.TemporaryDirectory() as td:
    project = RadarProject.create('PhaseD_Test', td)
    print(f'Project: {project.name} at {project.root}')
    
    # Verify profiles accessible
    smoke = project.profiles.get('smoke_v1')
    print(f'Profile: {smoke.name}')
    
    # Verify validation
    report = smoke.validate()
    print(f'Validation: success={report.success}')
    print(report.summary())

## 2. CaptureCollection API

In [ ]:
# PENDING: After Phase D implementation, test:
# project.captures.list()
# project.captures.latest()
# project.captures.get('...')
# for cap in project.captures: ...
# project.captures()  # deprecated compat
print('CaptureCollection API: pending Phase D implementation')

## 3. Session Semantics (Offline Structural Check)

In [ ]:
# PENDING: After Phase D implementation, test:
# session = project.connect()
# assert session.state == 'OPEN'
# session.close()
# assert session.state == 'CLOSED'
print('Session semantics: pending Phase D implementation')

## 4. Global Lock Model

In [ ]:
# PENDING: After Phase D implementation, test:
# Verify lock file created under LOCALAPPDATA\awr2944-dca\locks\
# Verify lock contains PID, hostname, endpoints
# Verify lock released on session.close()
# Verify stale lock detection
print('Global lock model: pending Phase D implementation')

## 5. Facade Delegation Order (Offline Dry-Run)

In [ ]:
# Verify the pre-hardware ordering works offline:
# Step 1: Resolve profile
smoke = RadarProfile.smoke_v1()
print(f'Step 1 - Profile resolved: {smoke.name}')

# Step 2: Validate
report = smoke.validate()
report.raise_for_errors()
print(f'Step 2 - Validation passed')

# Step 3: Compile SDK commands
effective = smoke.with_frame(frame_count=9)
cmds = effective.to_sdk_cli()
assert 'sensorStart' not in ' '.join(cmds)
assert 'sensorStop' not in ' '.join(cmds)
print(f'Step 3 - Compiled {len(cmds)} commands, no sensorStart/Stop')

# Step 4: Calculate byte plan
plan = effective.byte_plan(guard_frames=1)
print(f'Step 4 - Byte plan: {plan}')

# Step 5-10: Require hardware, deferred to section 9
print('Steps 5-10: deferred to live hardware test')

## 6. Override Precedence

In [ ]:
# Verify effective profile is immutable and frames override works
smoke = RadarProfile.smoke_v1()
assert smoke.frame.frame_count == 8

# Override frames
effective = smoke.with_frame(frame_count=16)
assert effective.frame.frame_count == 16

# Original unchanged
assert smoke.frame.frame_count == 8
print('Effective profile immutability confirmed')

# guard_frames is NOT part of the profile
assert not hasattr(smoke, 'guard_frames')
print('guard_frames correctly excluded from RadarProfile')

## 7. Unsupported Profile Fails Before Hardware

In [ ]:
from awr2944_dca.api.profile import ProfileCompilationNotSupported

bad = smoke.with_rf(slope_mhz_per_us=50.0)
try:
    bad.to_sdk_cli()
    print('FAIL: should have rejected')
except ProfileCompilationNotSupported as e:
    print(f'Correctly rejected before hardware: {len(e.unsupported_fields)} unsupported field(s)')
    for f in e.unsupported_fields:
        print(f'  {f["path"]}: expected={f["expected"]}, actual={f["actual"]}')

## 8. CaptureRawData & Verification Model (Structural)

In [ ]:
# PENDING: After Phase D implementation, test with an existing capture:
# capture.raw.native_path
# capture.raw.canonical_path
# capture.raw.to_cube(kind='canonical').shape
# capture.raw.native_sha256
# capture.verify()  -> CaptureVerificationReport
# capture.verify(strict=True)
print('CaptureRawData & Verification: pending Phase D implementation')

---

## 9. Live Hardware: Two-Capture Acceptance (REQUIRES HARDWARE)

**Do not run this section without live hardware connected.**

In [ ]:
SKIP_HARDWARE = True  # Set to False when running with live hardware

if not SKIP_HARDWARE:
    project = RadarProject.open(PROJECT_ROOT)
    
    # Pre-flight
    report = project.doctor()
    report.print()
    report.raise_for_errors(strict=True)
    
    # Two captures under one session
    with project.connect() as session:
        first = session.capture.run(
            profile='smoke_v1', frames=9, guard_frames=1,
            preserve_packet_metadata=True
        )
        second = session.capture.run(
            profile='smoke_v1', frames=9, guard_frames=1,
            preserve_packet_metadata=True
        )
    
    for label, result in [('FIRST', first), ('SECOND', second)]:
        m = result.session_result.manifest
        print(f'\n=== {label} CAPTURE ===')
        print(f'success: {result.success}')
        print(f'native_bytes: {m.native_byte_count}')
        print(f'canonical_bytes: {m.canonical_native_byte_count}')
        print(f'packet_records: {m.packet_record_count}')
        print(f'sequence_gaps: {m.sequence_gaps}')
        print(f'byte_counter_discontinuities: {m.byte_counter_discontinuity_count}')
        print(f'missing_bytes: {m.missing_payload_bytes}')
        print(f'overlap_bytes: {m.overlap_payload_bytes}')
        print(f'cube_shape: {m.logical_cube_shape}')
        
        assert result.success
        assert m.native_byte_count == 4_718_592
        assert m.canonical_native_byte_count == 4_194_304
        assert m.packet_record_count == 3_241
        assert m.sequence_gaps == 0
        assert m.byte_counter_discontinuity_count == 0
        assert m.missing_payload_bytes == 0
        assert m.overlap_payload_bytes == 0
        assert m.logical_cube_shape == [8, 128, 4, 256]
        
        # Verify and cube
        result.capture.verify(strict=True)
        cube = result.capture.raw.to_cube()
        assert cube.shape == (8, 128, 4, 256)
        print(f'Verification and cube PASSED')
        
        # Record SHA256 for provenance
        print(f'native_sha256: {m.native_sha256}')
        print(f'canonical_sha256: {m.canonical_sha256}')
    
    print('\n=== TWO-CAPTURE HARDWARE ACCEPTANCE PASSED ===')
else:
    print('Hardware test skipped (SKIP_HARDWARE=True)')

## 10. Emergency Stop (REQUIRES HARDWARE)

In [ ]:
if not SKIP_HARDWARE:
    stop_report = project.emergency_stop()
    print(f'Radar stop: {stop_report.radar_stop}')
    print(f'DCA stop: {stop_report.dca_stop}')
else:
    print('Emergency stop test skipped (SKIP_HARDWARE=True)')

## Final Marker

In [ ]:
print('PHASE_D_SESSION_AND_CAPTURE_FACADE_PASS')